# Autonomous Vehicle Sensor Integrity: ML-Based Spoof Detection & Physical Cause Attribution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nithesh-hash/av-sensor-integrity/blob/main/ml/train.ipynb)

This notebook trains two **Random Forest Classifiers** ($n\_estimators=50, max\_depth=8$) on synthetic sensor integrity telemetry:
1. **Binary Spoof Detector (`is_spoof`)**: Distinguishes adversarial velocity manipulation from benign kinematic artifacts.
2. **Multiclass Cause Attribution (`cause`)**: Classifies the root physical phenomenon (e.g. MTI Clutter Notch, Radar RCS Multipath Dropout, Guardrail Multi-bounce Ghost, Boresight Calibration Drift, Optical Velocity Injection, or Consistent).

Trained decision trees are subsequently exported to `model.json` for real-time, low-latency TypeScript client-side evaluation.

## 1. Environment Setup & Data Loading

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GroupShuffleSplit

# Load dataset.csv (upload to Colab working directory or load locally)
csv_path = 'dataset.csv' if os.path.exists('dataset.csv') else '../dataset.csv'
df = pd.read_csv(csv_path)
print(f"Loaded dataset with {len(df):,} rows and {len(df.columns)} columns.")
df.head()

## 2. Dataset Inspection & Group Splitting
To avoid data leakage across consecutive frames of the same simulation run, we perform a **GroupShuffleSplit** on `run_id`.

In [ ]:
feature_cols = [
    'residual',
    'sigmaTotal',
    'mahalanobisDist',
    'vExpectedRad',
    'cosPhi',
    'totalSpeed',
    'lidarDensity',
    'radarSnr',
    'radarDetected',
    'trackAge',
    'rangeElongation',
    'trustScore'
]

print("Features:", feature_cols)
print("\nClass Distribution (cause):")
print(df['cause'].value_counts())

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['run_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"\nTrain set: {len(train_df):,} rows ({train_df['run_id'].nunique()} unique runs)")
print(f"Test set:  {len(test_df):,} rows ({test_df['run_id'].nunique()} unique runs)")

X_train = train_df[feature_cols].values
X_test = test_df[feature_cols].values

y_spoof_train = train_df['is_spoof'].values
y_spoof_test = test_df['is_spoof'].values

y_cause_train = train_df['cause'].values
y_cause_test = test_df['cause'].values

## 3. Binary Spoof Detector (`is_spoof`)
Train `RandomForestClassifier(n_estimators=50, max_depth=8)` to detect attacks while guaranteeing false alarm rate (FAR) < 2% on benign traffic.

In [ ]:
clf_spoof = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
clf_spoof.fit(X_train, y_spoof_train)

y_pred_spoof = clf_spoof.predict(X_test)
acc_spoof = accuracy_score(y_spoof_test, y_pred_spoof)

# Compute Benign False Alarm Rate (FAR) and Attack Detection Rate
benign_mask = (y_spoof_test == 0)
false_alarms = np.sum((y_pred_spoof == 1) & benign_mask)
total_benign = np.sum(benign_mask)
far_benign = (false_alarms / total_benign) * 100.0

attack_mask = (y_spoof_test == 1)
true_detections = np.sum((y_pred_spoof == 1) & attack_mask)
total_attacks = np.sum(attack_mask)
det_rate = (true_detections / total_attacks) * 100.0

print(f"=== Binary Spoof Classifier Evaluation ===")
print(f"Accuracy:                {acc_spoof * 100:.2f}%")
print(f"Benign False Alarm Rate: {far_benign:.2f}% ({false_alarms}/{total_benign})")
print(f"Attack Detection Rate:   {det_rate:.2f}% ({true_detections}/{total_attacks})")

cm_spoof = confusion_matrix(y_spoof_test, y_pred_spoof)
print("\nConfusion Matrix (Binary Spoof Detection):")
print(f"                Pred: Benign (0)  Pred: Spoof (1)")
print(f"Actual: Benign    {cm_spoof[0, 0]:<17} {cm_spoof[0, 1]:<17}")
print(f"Actual: Spoof     {cm_spoof[1, 0]:<17} {cm_spoof[1, 1]:<17}")

### Feature Importances for Binary Spoof Detection

In [ ]:
spoof_imp = pd.Series(clf_spoof.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
spoof_imp.plot(kind='barh', color='#3b82f6')
plt.title("Feature Importances - Binary Spoof Detection")
plt.xlabel("Gini Importance")
plt.tight_layout()
plt.show()

for feat, imp in spoof_imp.sort_values(ascending=False).items():
    print(f"{feat:<18}: {imp:.4f}")

## 4. Multiclass Cause Attribution Model (`cause`)
Train `RandomForestClassifier(n_estimators=50, max_depth=8)` to attribute the physical cause across all 10 scenario categories.

In [ ]:
clf_cause = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
clf_cause.fit(X_train, y_cause_train)

y_pred_cause = clf_cause.predict(X_test)
acc_cause = accuracy_score(y_cause_test, y_pred_cause)

print(f"=== Multiclass Cause Classifier Evaluation ===")
print(f"Accuracy: {acc_cause * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_cause_test, y_pred_cause, digits=4))

cause_classes = clf_cause.classes_.tolist()
cm_cause = confusion_matrix(y_cause_test, y_pred_cause, labels=cause_classes)
cm_df = pd.DataFrame(cm_cause, index=cause_classes, columns=cause_classes)
print("\nConfusion Matrix (Multiclass Cause):")
cm_df

### Feature Importances for Cause Attribution

In [ ]:
cause_imp = pd.Series(clf_cause.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
cause_imp.plot(kind='barh', color='#10b981')
plt.title("Feature Importances - Physical Cause Attribution")
plt.xlabel("Gini Importance")
plt.tight_layout()
plt.show()

for feat, imp in cause_imp.sort_values(ascending=False).items():
    print(f"{feat:<18}: {imp:.4f}")

## 5. Export Decision Forests to `model.json`
Extract node-level feature indices, split thresholds, child pointers, and class probability distributions for in-browser TypeScript inference.

In [ ]:
def serialize_forest(rf):
    serialized_trees = []
    for est in rf.estimators_:
        t = est.tree_
        raw_val = t.value[:, 0, :]
        sums = raw_val.sum(axis=1, keepdims=True)
        sums[sums == 0] = 1.0
        probs = np.round(raw_val / sums, 4).tolist()

        serialized_trees.append({
            'feature': t.feature.tolist(),
            'threshold': np.round(t.threshold, 4).tolist(),
            'left': t.children_left.tolist(),
            'right': t.children_right.tolist(),
            'values': probs,
        })
    return {
        'classes': [int(c) if isinstance(c, (np.integer, int)) else str(c) for c in rf.classes_],
        'n_estimators': len(rf.estimators_),
        'max_depth': rf.max_depth,
        'trees': serialized_trees,
    }

model_payload = {
    'version': '1.0.0',
    'features': feature_cols,
    'spoof_forest': serialize_forest(clf_spoof),
    'cause_forest': serialize_forest(clf_cause),
    'metrics': {
        'spoof_accuracy': round(float(acc_spoof), 4),
        'spoof_far': round(float(far_benign), 4),
        'spoof_detection_rate': round(float(det_rate), 4),
        'cause_accuracy': round(float(acc_cause), 4),
    }
}

out_json = 'model.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(model_payload, f)
print(f"Successfully exported {out_json} ({os.path.getsize(out_json) / 1024:.1f} KB)")